# How to Leverage Spacy Rules NER
> This blog post outlines the important features in Spacy Rules NER
- toc: true
- branch: master
- author: Senthil Kumar
- badges: true
- comments: true
- categories: [spacy/NER]
- image: images/spacy/spacy_nlp_pipeline.png
- hide: false

## Introduction to SpaCy and NER

### About SpaCy
- SpaCy is a NLP library offering easy-to-use Python API for many information extraction and machine learning tasks in text data
- They are internally written in Cython and hence occupies low memory foot print with its `small` models and are quite fast with decent accuracy



Source:
- [more about spaCy](https://spacy.io/)

### What is NER

> Named-entity recognition (NER) (also known as (named) entity identification, entity chunking, and entity extraction) is `a subtask of information extraction` that seeks to locate and classify named entities mentioned in unstructured text into pre-defined categories

Source: [Wikipedia Article on NER](https://en.wikipedia.org/wiki/Named-entity_recognition)

> In information extraction, a named entity is a real-world object, such as a person, location, organization, product, etc., that can be denoted with a proper noun. <br>
> For example, in the sentence - "Biden is the president of the United States", <br>
> "Biden" and "the United States" are `named entities` (proper nouns). "president" is not a named entity

Source: [Wikipedia Article on Named Entities](https://en.wikipedia.org/wiki/Named_entity)

### The Spacy Version used here

In [2]:
#collapse_show
!python3 -m spacy validate

✔ Loaded compatibility table

================= Installed pipeline packages (spaCy v3.1.3) =================
ℹ spaCy installation: /usr/local/lib/python3.7/dist-packages/spacy

NAME             SPACY            VERSION                            
en_core_web_sm   >=3.1.0,<3.2.0   3.1.0   ✔



In [3]:
#collapse_show
# goal: to capture Car Models to their corresponding Makes
sample_sentence_1 = "Go for ford mustang mache variants. Mustang has a deceivingly huge trunk and good horse power. If you want reliability, Toyota Lexus should be your choice. Lexus has good body style too"
sample_sentence_2 = "Considering the harsh winters here, I am considering 2014 Nissan Murano or the '14 Subaru Forester"
sample_sentence_3 = "Among used cars, I am still not sure what to choose - Civic or Corolla?"

sample_sentences = [sample_sentence_1, 
                    sample_sentence_2,
                    sample_sentence_3
                   ]


## 1. Basic Featues of SpaCy NER

### Token Matcher and Phrase Matcher

In [6]:
#collapse-hide
import spacy
from spacy.matcher import Matcher, PhraseMatcher
from spacy.tokens import Span
from spacy.util import filter_spans
from spacy import displacy

nlp = spacy.load('en_core_web_sm',disable=['ner'])
matcher = Matcher(nlp.vocab)

# pattern rules matching every token
ford_pattern = [{"LOWER": "ford", "OP":"?"},
                   {"LOWER": "mustang"},
                   {"LOWER":{"IN":["mache","gt","bulitt"]},"OP":"*"}
                  ]
toyota_pattern = [{"LOWER": "toyota","OP":"?"},
                 {"LOWER": {"IN":["lexus","corolla","camry"]}}
                ]

honda_pattern = [{"LOWER": "honda","OP":"?"},
                 {"LOWER": {"IN":["civic","accord"]}}
                ]

token_matcher_patterns = {"FORD": ford_pattern,
                          "TOYOTA": toyota_pattern,
                          "HONDA": honda_pattern
                         }

# phrase pattern looks for exact match
nissan_phrase_pattern = ["Nissan Murano", "Murano", "murano", "nissan murano"]
subaru_phrase_pattern = ["Subaru Forester", "Forester", "forester", "subaru forester"]

phrase_matcher_patterns = {"NISSAN": nissan_phrase_pattern,
                           "SUBARU": subaru_phrase_pattern
                          }

def add_token_matcher_and_phrase_matcher_patterns(nlp_model,
                                                  token_patterns_dict=token_matcher_patterns,
                                                  phrase_patterns_dict=phrase_matcher_patterns
                                                 ):
    token_matcher = Matcher(nlp_model.vocab)
    for key, value in token_patterns_dict.items():
        token_matcher.add(key,[value])
        
    phrase_matcher = PhraseMatcher(nlp_model.vocab)
    for key, terms_list in phrase_patterns_dict.items():
        phrase_patterns = [nlp_model.make_doc(text) for text in terms_list]
        phrase_matcher.add(key, phrase_patterns)
    return token_matcher, phrase_matcher

doc = nlp(sample_sentence_1)

def modify_doc(token_matcher,
               phrase_matcher,
               doc):
    original_ents = list(doc.ents)
    matches = token_matcher(doc) + phrase_matcher(doc)
    for match_id, start, end in matches:
        span = Span(doc, start, end, match_id)
        original_ents.append(span)
    filtered = filter_spans(original_ents)
    doc.ents = filtered
    return doc

In [7]:
# collapse-show
token_matcher, phrase_matcher = add_token_matcher_and_phrase_matcher_patterns(nlp, 
                                                                              token_matcher_patterns,
                                                                              phrase_matcher_patterns
                                                                             )
for doc in nlp.pipe(sample_sentences,
             as_tuples=False
            ):
    new_doc = modify_doc(token_matcher,
                         phrase_matcher,
                         doc
                        )
    displacy.render(new_doc,
                    style='ent')
    print("*****")

*****


*****


*****


## 2. Externalizing Rules from Codes

### Entity Ruler allowing token-based and exact phrase rules

## 3. Custom Components in Spacy Pipeline

### Adding RegEx patterns as a custom component
### Chaining NER

## 4. How to save Rules NER as a package

## 5. Conclusion

## 6. ToDos
> The following topics are evolving. I will keep adding material as I find useful 
### 6A. How to use Spacy's Dependency Matcher
### 6B. Attempting Nested NER as two NER pipeline components on the same data

## References

- Spacy Rules based Matching | [link](https://spacy.io/usage/rule-based-matching)